# Classification d’images avec Fashion-MNIST

## Objectif

Nous allons construire progressivement un réseau convolutif, ou CNN,
capable de reconnaître des vêtements représentés par des images.

Le travail suivra toujours le même cycle :

1. comprendre et préparer les données ;
2. construire une architecture adaptée ;
3. entraîner le modèle ;
4. surveiller la validation ;
5. évaluer une seule fois sur le test ;
6. analyser les réussites et les erreurs.

Pour chaque étape, la théorie et les formules mathématiques seront
présentées avant le code.

Dans cette première étape, nous allons uniquement :

- préparer l’environnement ;
- comprendre comment une image est représentée par des nombres ;
- charger Fashion-MNIST ;
- vérifier les dimensions des données.

Nous ne normaliserons pas encore les images et nous ne construirons pas
encore le CNN.

## 1. Environnement et reproductibilité

NumPy permet de manipuler les tableaux numériques. Matplotlib permet
d’afficher les images. TensorFlow et Keras permettront de charger les
données et de construire le futur CNN.

Certaines opérations du Deep Learning utilisent des valeurs aléatoires :

- initialisation des poids ;
- mélange des observations ;
- séparation entraînement-validation ;
- Dropout.

Nous fixons donc une graine :

\[
\text{SEED}=42
\]

Si deux opérations utilisent la même méthode et la même graine, elles
produisent généralement la même suite pseudo-aléatoire.

Cela améliore la reproductibilité, mais des différences peuvent encore
apparaître entre plusieurs versions de TensorFlow, plusieurs systèmes
d’exploitation ou plusieurs matériels.

In [1]:
import random

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow :", tf.__version__)
print("NumPy      :", np.__version__)
print("Environnement prêt")

TensorFlow : 2.21.0
NumPy      : 2.4.6
Environnement prêt


## 2. Une image est une matrice de nombres

Une image Fashion-MNIST mesure 28 pixels de hauteur et 28 pixels de
largeur.

Comme l’image est en niveaux de gris, chaque pixel est représenté par un
seul nombre entier compris entre 0 et 255 :

\[
p_{r,c}\in\{0,1,\ldots,255\}
\]

où :

- \(r\) désigne la ligne du pixel ;
- \(c\) désigne sa colonne ;
- \(0\) représente le noir ;
- \(255\) représente le blanc ;
- les valeurs intermédiaires représentent différentes nuances de gris.

Une image est donc une matrice :

\[
I\in\{0,\ldots,255\}^{28\times28}
\]

Le nombre total de pixels est :

\[
28\times28=784
\]

Une matrice est un tableau d’ordre 2. Lorsque nous regroupons \(N\)
images, nous obtenons un tenseur d’ordre 3 :

\[
X\in\{0,\ldots,255\}^{N\times28\times28}
\]

Les trois dimensions représentent :

1. le numéro de l’image ;
2. la ligne du pixel ;
3. la colonne du pixel.

Le canal de niveaux de gris est encore implicite. Avant d’utiliser un
CNN, nous pourrons ajouter une dimension :

\[
(N,28,28)
\longrightarrow
(N,28,28,1)
\]

Le dernier \(1\) indiquera qu’il existe un seul canal.

## 3. Classification supervisée multiclasse

Chaque image \(X_i\) possède une étiquette \(y_i\) indiquant la catégorie
du vêtement :

\[
y_i\in\{0,1,\ldots,9\}
\]

Une observation supervisée est donc un couple :

\[
(X_i,y_i)
\]

L’objectif futur sera d’apprendre une fonction paramétrée par
\(\theta\) :

\[
f_\theta(X_i)
=
\widehat{\mathbf p}_i
\]

où :

\[
\widehat{\mathbf p}_i
=
\begin{pmatrix}
\widehat p_{i,0}\\
\widehat p_{i,1}\\
\vdots\\
\widehat p_{i,9}
\end{pmatrix}
\]

contiendra les probabilités prédites pour les dix classes.

Ces probabilités devront vérifier :

\[
0\leq\widehat p_{i,k}\leq1
\]

et :

\[
\sum_{k=0}^{9}\widehat p_{i,k}=1
\]

La classe prédite sera celle ayant la plus grande probabilité :

\[
\widehat y_i
=
\underset{k}{\operatorname{argmax}}
\left(\widehat p_{i,k}\right)
\]

Il s’agit d’une classification multiclasse exclusive : chaque image
appartient à une seule classe parmi dix.

## 4. Ensembles d’entraînement, de validation et de test

Keras fournit Fashion-MNIST sous la forme de deux ensembles officiels :

- 60 000 images d’entraînement ;
- 10 000 images de test.

Nous ne devons pas utiliser le test pour choisir l’architecture, le
nombre de couches, le nombre d’époques ou les hyperparamètres.

Plus tard, nous extrairons une validation uniquement depuis les
60 000 images d’entraînement :

\[
D_{\text{train complet}}
\longrightarrow
D_{\text{train}} \cup D_{\text{validation}}
\]

Le test restera indépendant :

\[
D_{\text{test}}
\cap
D_{\text{train}}
=
\varnothing
\]

\[
D_{\text{test}}
\cap
D_{\text{validation}}
=
\varnothing
\]

À cette étape, nous allons seulement charger le test et vérifier sa
structure. Nous ne l’utiliserons ni pour entraîner un modèle ni pour
prendre une décision.

In [2]:
(
    (
        images_train_complet_brutes,
        etiquettes_train_complet,
    ),
    (
        images_test_brutes,
        etiquettes_test,
    ),
) = tf.keras.datasets.fashion_mnist.load_data()

print("Fashion-MNIST chargé.")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 1s 19us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 40s 2us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Fashion-MNIST chargé.


## 5. Formes attendues des tenseurs

Pour l’entraînement complet, nous attendons :

\[
X_{\text{train complet}}
\in
\{0,\ldots,255\}^{60000\times28\times28}
\]

et :

\[
y_{\text{train complet}}
\in
\{0,\ldots,9\}^{60000}
\]

Pour le test, nous attendons :

\[
X_{\text{test}}
\in
\{0,\ldots,255\}^{10000\times28\times28}
\]

et :

\[
y_{\text{test}}
\in
\{0,\ldots,9\}^{10000}
\]

La première dimension de \(X\) doit être égale à la longueur de \(y\),
car chaque image doit posséder exactement une étiquette.

Les images sont encore stockées sous forme de nombres entiers `uint8`.
Le terme signifie « entier non signé codé sur 8 bits ».

Avec 8 bits, le nombre de valeurs possibles est :

\[
2^8=256
\]

ce qui correspond exactement aux valeurs comprises entre 0 et 255.

In [3]:
print(
    "Images d’entraînement :",
    images_train_complet_brutes.shape,
)

print(
    "Étiquettes d’entraînement :",
    etiquettes_train_complet.shape,
)

print(
    "Images de test :",
    images_test_brutes.shape,
)

print(
    "Étiquettes de test :",
    etiquettes_test.shape,
)

print(
    "\nType des images :",
    images_train_complet_brutes.dtype,
)

print(
    "Nombre de pixels par image :",
    images_train_complet_brutes.shape[1]
    * images_train_complet_brutes.shape[2],
)

assert images_train_complet_brutes.shape == (
    60_000,
    28,
    28,
)

assert etiquettes_train_complet.shape == (
    60_000,
)

assert images_test_brutes.shape == (
    10_000,
    28,
    28,
)

assert etiquettes_test.shape == (
    10_000,
)

assert len(images_train_complet_brutes) == len(
    etiquettes_train_complet
)

assert len(images_test_brutes) == len(
    etiquettes_test
)

print("\nLes dimensions sont cohérentes.")

Images d’entraînement : (60000, 28, 28)
Étiquettes d’entraînement : (60000,)
Images de test : (10000, 28, 28)
Étiquettes de test : (10000,)

Type des images : uint8
Nombre de pixels par image : 784

Les dimensions sont cohérentes.
